### This script demonstrates how dithering can be applied to low-amplitude signals that are vulnerable to quantization error. After setting the variables, you may click "Runtime > Run all" (if you use Colab) or click the "Run All" button (if you use Visual Studio Code) to run all cells in one operation.

## Procedures
1. Read the codes in this .ipynb file to understand how they work.
1. set *use_colab* to True if you use Google Colab; otherwise, set it to False.
1. Set *source* to 'tone'.   
1. Run all cells. Listen to the audio of (1) the original signal, (2) the quantized version of the amplitude-reduced signal, and (3) the quantized version of the amplitude-reduced signal after adding noise (dithering).
1. Repeat Steps 3 and 4 by setting *source* to 'music'.
1. Change the constant *n_bits* to 4 and repeat Steps 3-5.
1. Vary other variables to see their effect on the audio quality.


## Answer the following questions
1. When the signal amplitude is high enough (e.g., without any amplitude reduction), the quantized signal _xq_ is perceptually close to the original signal "x" even at a bit depth of 8. Why is this the case?
1. When using a low-precision quantizer, the quantized waveform _yq_ does not look like the original waveform _y_. Why is this the case?
1. When the bit-depth is 4 (*n_bits = 4*), the quantization error becomes periodic. How will this kind of quantization error affect the audio quality of the quantized signal?
1. Why does adding a small amount of noise to the signal that is vulnerable to quantization error help to reduce the detrimental effect of the quantization error?
1. When *source* is set to 'tone', explain why severe quantization noise can change the perceptual frequency of the tone. You may check this phenomenon by listening to the audio of _yq_.


## Program Codes ##

In [ ]:
import matplotlib.pyplot as plt
import librosa
import numpy as np
import IPython
from IPython.display import Audio

In [ ]:
# Define some variables
source = 'tone'            # Either 'tone' or 'music'
use_colab = True            # Set to False if you use local IDE such as Visual Studio Code
sr = 16000                  # Sampling rate of the target signal in Hz
n_bits_h = 16               # Bit-depth for high-precision quantization
n_bits_l = 4                # Bit-depth for low-precision quantization

In [ ]:
# Mount Google Drive and create a folder for the lab exercise
if use_colab == True:
  from google.colab import drive
  drive.mount('/content/drive')
  !mkdir -p /content/drive/MyDrive/Learning/EIE4435/Programs
  %cd /content/drive/MyDrive/Learning/EIE4435/Programs

In [ ]:
# Function for plotting one frame of unquantized waveform, one frame of quantized waveform, and their spectrograms
def plot_figure(s, sq, sr=16000, frm_len=512, hop_len=256):
    e = sq - s
    S = librosa.amplitude_to_db(np.abs(librosa.stft(s, n_fft=512, hop_length=256)), ref=np.max)  # STFT of s
    Sq = librosa.amplitude_to_db(np.abs(librosa.stft(sq, n_fft=512, hop_length=256)), ref=np.max)  # STFT of sq
    E = librosa.amplitude_to_db(np.abs(librosa.stft(e, n_fft=512, hop_length=256)), ref=np.max)  # STFT of e
    fig, ax = plt.subplots(nrows=3, sharex=True, figsize=(8,3))
    smp_idx = range(2*frm_len, 3*frm_len)
    librosa.display.waveshow(s[smp_idx], sr=sr, ax=ax[0])
    ax[0].set_ylabel('Original')
    librosa.display.waveshow(sq[smp_idx], sr=sr, ax=ax[1])
    ax[1].set_ylabel('Quantized')
    librosa.display.waveshow(e[smp_idx], sr=sr, ax=ax[2])
    ax[2].set_ylabel('Error')
    _, ax = plt.subplots(nrows=3, sharex=True, figsize=(8,3))
    librosa.display.specshow(S, sr=sr, n_fft=frm_len, hop_length=hop_len, x_axis='time', y_axis='linear', ax=ax[0])
    ax[0].set_ylabel('Original')
    librosa.display.specshow(Sq, sr=sr, n_fft=frm_len, hop_length=hop_len, x_axis='time', y_axis='linear', ax=ax[1])
    ax[1].set_ylabel('Quantized')
    librosa.display.specshow(E, sr=sr, n_fft=frm_len, hop_length=hop_len, x_axis='time', y_axis='linear', ax=ax[2])
    ax[2].set_ylabel('Error')

In [ ]:
# Function for performing quanitzation, default 8-bit quantization and max amplitude being 1.0
def quantize(x, n_bits=8, xmax=1):
    Q = 2 * xmax/(2**n_bits)                        # Quantization level
    xq = x/2 * (2**(n_bits-1) + 2**(n_bits-1) - 1)
    xq = np.floor(xq).astype(int).astype(float)*Q
    print(f"{n_bits}-bit audio ranges from -{2**(n_bits - 1)*Q:.3f} to {(2**(n_bits - 1) - 1)*Q:.3f} for xmax={xmax} and Q={Q:.3e}")
    print(f"Max quantized value: {np.max(xq):.3f}, Avg quantized value: {np.mean(xq):.2f}")
    return xq

In [ ]:
# Check a 3-bit quantizer with xmax=1
y = np.array([-1, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])    # y ranges from -xmax to xmax
yq = quantize(y, n_bits=3, xmax=1)
np.set_printoptions(precision=2, floatmode='fixed')
print(f"y =  {y}")
print(f"yq = {yq}")
print(f"e =  {yq-y}")

In [ ]:
# Function for playing back (1) original signal, (2) undithered quantized signal,
# and (3) dithered quantized signal
def audio_playback(original, undithered_quantized, dithered_quantized=[], sr=16000):
  print('Original:')
  IPython.display.display(Audio(data=original, rate=sr))
  print('Undithered quantized:')
  IPython.display.display(Audio(data=undithered_quantized, rate=sr))
  if len(dithered_quantized) != 0:
    print('Dithered quantized:')
    IPython.display.display(Audio(data=dithered_quantized, rate=sr))

In [ ]:
if source == 'music':
    # Load a wavefile and crop a 2-second segment from the average of the left and right channels
    # Then, reduce the sampling rate to 16 kHz.
    audiofile = '../AudioFiles/Bach.mp3'
    x, fs = librosa.load(audiofile, mono=True, offset=0.0, duration=2)
    x = librosa.resample(x, orig_sr=fs, target_sr=sr)
    print(f'Audio file {audiofile} loaded')
else:
    print('Generate a tone of 500Hz with a sampling rate of 16kHz and duration of 1 second')
    x = librosa.tone(500, sr=sr, length=sr)

# rescale the samples to the range -1 and +1.
x = ((x-np.min(x))/(np.max(x)-np.min(x))-0.5)*2

In [ ]:
# Quantize the segment using high-precision quantization (16 bits) and plot the waveform
xq = quantize(x, n_bits=n_bits_h)
plot_figure(x, xq, sr=sr)

In [ ]:
# Reduce the amplitude of the signal to make it more vulnerable to quanization error.
# Then, quantize the low-amplitude signal with a low-precision (3-bit) quantizer to
# exaggerate the issue of quantization error on low-amplitude signals.
y = x/2.0
yq = quantize(y, n_bits=n_bits_l, xmax=1)

# Plot a short segment of the low-amplitude waveform, quantized waveform, quantization error, and their spectrograms
plot_figure(y, yq, sr=sr)

# Create playback buttons to compare the original and quantized signals
audio_playback(y, yq)

In [ ]:
# Implement a triangular ditherer
class TriangularDitherer:
    """
    Triangular ditherer for audio signals.
    PDF spans from -Q to +Q with maximum density of 1/Q at the center.
    """

    def __init__(self, Q=1.0, bit_depth=16):
        """
        Initialize the triangular ditherer.

        Parameters:
        -----------
        Q : float
            Quantization step size (LSB value)
        bit_depth : int
            Audio bit depth (for quantization)
        """
        self.Q = Q
        self.bit_depth = bit_depth
        self.max_amplitude = 2 ** (bit_depth - 1)

    def generate_dither(self, num_samples):
        """
        Generate triangular dither samples.

        The triangular distribution is created by summing two independent
        uniform distributions (Irwin-Hall distribution of order 2).

        Parameters:
        -----------
        num_samples : int
            Number of dither samples to generate

        Returns:
        --------
        dither : numpy.ndarray
            Array of dither samples with triangular PDF spanning [-Q, +Q]
        """
        # Generate triangular dither by summing two uniform random variables
        # Each uniform spans [-Q/2, +Q/2], so their sum spans [-Q, +Q]
        u1 = np.random.uniform(-self.Q/2, self.Q/2, num_samples)
        u2 = np.random.uniform(-self.Q/2, self.Q/2, num_samples)

        # Sum of two uniforms gives triangular distribution
        dither = u1 + u2

        return dither

    def apply_dithering(self, audio_signal):
        """
        Apply triangular dithering to an audio signal followed by quantization.

        Parameters:
        -----------
        audio_signal : numpy.ndarray
            Input audio signal (float values)

        Returns:
        --------
        dithered_quantized : numpy.ndarray
            Dithered and quantized audio signal
        dither : numpy.ndarray
            The dither signal used
        """
        # Generate dither
        dither = self.generate_dither(len(audio_signal))

        # Add dither to the audio signal
        dithered_signal = audio_signal + dither

        # Quantize the dithered signal
        dithered_quantized = self.quantize(dithered_signal)

        return dithered_quantized, dither

    def quantize(self, signal):
        """
        Quantize the signal to the specified bit depth.

        Parameters:
        -----------
        signal : numpy.ndarray
            Input signal to quantize

        Returns:
        --------
        quantized : numpy.ndarray
            Quantized signal
        """
        # Scale to integer range
        scaled = signal * (self.max_amplitude / self.Q)

        # Round to nearest integer (mid-tread quantizer)
        quantized = np.round(scaled)

        # Scale back to original range
        quantized = quantized * (self.Q / self.max_amplitude)

        return quantized

In [ ]:
# Apply triangular dithering and quantize the the noise added signal using a
# low-precision quantizer.
Q = 2/(2**n_bits_l)     # Signal range of x is -1 to 1, regardless of the range of y
ditherer = TriangularDitherer(Q=Q, bit_depth=n_bits_l)
dithered_quantized, dither = ditherer.apply_dithering(y)  # Dithering and quantization

# Plot a frame of dithered signal, the quantization error, and their spectrograms
plot_figure(y, dithered_quantized, sr=sr)

# Create buttons for comparing audio quality
yq = quantize(y, n_bits=n_bits_l, xmax=1)
audio_playback(y, yq, dithered_quantized=dithered_quantized)

## **Summary Grade Sheet**

| Item | Max Points
| :--- | :--- |
| Correctness of answers   | 20 |
| Explanation of answers   | 20 |  
| Use of figures for explanations | 20 |
| Sufficient exploration (e.g., change of variables) | 20 |
| Clarify of writing   | 20 |
